In [1]:
from groq import Groq
from dotenv import load_dotenv

load_dotenv()


client = Groq()


In [2]:
weather_db={
    "london":{"celcius":22,"sky":"cloudy"},
    "paris":{"celcius":24,"sky":"sunny"},
    "moscow":{"celcius":10,"sky":"rainy"},
    "delhi":{"celcius":31,"sky":"sunny"},
    "kanpur":{"celcius":29,"sky":"sunny"},
    "cairo":{"celcius":24,"sky":"rainy"},
    "mexico":{"celcius":20,"sky":"cloudy"},
    
}

In [3]:
def lookup_weather(location:str)->str:
    """look up the weather for a location"""
    record = weather_db.get(location.lower())
    if record is None:
        return f"sorry,i don't know the weather of {location}."
    return f"the weather in {location} is {record['celcius']}c and {record['sky']}"

In [6]:
# schema given to llm

weather_Schema={
    "type":"function",
    "function":{
        "name":"lookup_weather",
        "description":"lookup weather for local time",
        "parameters":{
            "type":"object",
            "properties":{
                "location":{"type":"string","description":"location to look up for weather of place"}
            },
            "required":["location"]
        }
    }
}

TOOLS={"lookup_weather":lookup_weather}

In [9]:
result = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {
            "role": "user",
            "content": "What is the weather in Delhi?"
        }
    ],
    tools=[weather_Schema],
   
)

print(result.choices[0].message)

ChatCompletionMessage(content=None, role='assistant', annotations=None, executed_tools=None, function_call=None, reasoning='The user asks for weather in Delhi. We can use the function lookup_weather with location "Delhi".', tool_calls=[ChatCompletionMessageToolCall(id='fc_d9fb75f7-e36e-4305-a177-f984b68e1572', function=Function(arguments='{"location":"Delhi"}', name='lookup_weather'), type='function')])


In [10]:

from groq import Groq
from dotenv import load_dotenv
import json

load_dotenv()
client = Groq()

# 1. Weather database
weather_db = {
    "london": {"celcius": 22, "sky": "cloudy"},
    "paris": {"celcius": 24, "sky": "sunny"},
    "moscow": {"celcius": 10, "sky": "rainy"},
    "delhi": {"celcius": 31, "sky": "sunny"},
    "kanpur": {"celcius": 29, "sky": "sunny"},
    "cairo": {"celcius": 24, "sky": "rainy"},
    "mexico": {"celcius": 20, "sky": "cloudy"},
}


# 2. Python function
def lookup_weather(location: str) -> str:
    """Look up the weather for a location."""
    record = weather_db.get(location.lower())

    if record is None:
        return f"Sorry, I don't know the weather of {location}."

    return (
        f"The weather in {location} is "
        f"{record['celcius']}°C and {record['sky']}."
    )


# 3. Tool schema given to the LLM
weather_Schema = {
    "type": "function",
    "function": {
        "name": "lookup_weather",
        "description": "Look up the weather for a location.",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "Location to look up weather for."
                }
            },
            "required": ["location"],
            "additionalProperties": False
        }
    }
}


# 4. Tools mapping
TOOLS = {
    "lookup_weather": lookup_weather
}


# 5. Send user query to the model
messages = [
    {
        "role": "user",
        "content": "What is the weather in Delhi?"
    }
]

result = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=messages,
    tools=[weather_Schema],
    tool_choice="auto"
)

response = result.choices[0].message

# 6. Execute tool call if requested
if response.tool_calls:

    # Add assistant's tool-call message to conversation
    messages.append(response)

    for tool_call in response.tool_calls:
        function_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)

        if function_name in TOOLS:
            tool_result = TOOLS[function_name](**arguments)
        else:
            tool_result = "Unknown function requested."

        print("Tool:", function_name)
        print("Arguments:", arguments)
        print("Tool result:", tool_result)

        # 7. Send tool output back to the model
        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": str(tool_result)
        })

    # 8. Get the final answer from the model
    final_result = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=messages
    )

    print("\nFinal Answer:")
    print(final_result.choices[0].message.content)

else:
    print("Final Answer:")
    print(response.content)


Tool: lookup_weather
Arguments: {'location': 'Delhi'}
Tool result: The weather in Delhi is 31°C and sunny.

Final Answer:
The weather in Delhi is 31 °C and sunny.
